<a href="https://colab.research.google.com/github/parkjw8/AIFFEL_Quest_ENG/blob/main/Main_Quest/MQ02/jeje.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q gdown ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 27.8 MB/s eta 0:00:00


In [2]:
# 2) 데이터 (전체본 2.6GB, 파일ID 직접 — --fuzzy 쓰지 말 것: gdown 신버전서 제거됨)
!gdown 1JreaY64UPq7ugrK6iClb99BMiH15cwUg -O NordTank.zip
!unzip -q NordTank.zip -d /content/
!ls /content/NordTank586x371   # images  labels 보이면 OK

Downloading...
From (original): https://drive.google.com/uc?id=1JreaY64UPq7ugrK6iClb99BMiH15cwUg
From (redirected): https://drive.google.com/uc?id=1JreaY64UPq7ugrK6iClb99BMiH15cwUg&confirm=t&uuid=3bb3e89c-2b08-43b9-a294-e22caabe37d5
To: /content/NordTank.zip
100% 2.76G/2.76G [00:26<00:00, 102MB/s]
images	labels


In [3]:
# 3) 구글 드라이브 마운트 (체크포인트 저장용 — 팝업서 권한 허용)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# 4) GPU 확인
import torch; print(torch.cuda.get_device_name(0))   # Tesla T4 나오면 OK

Tesla T4


In [5]:
"""
================================================================
 MQ02 풍력터빈 손상탐지 — 팀 공통 베이스 코드 (이 파일 하나면 됩니다)
================================================================
 하는 일:  데이터 split(자동)  →  내 모델 학습  →  결과 출력
 사용법 :  맨 위 [설정]에서 "MODEL"과 "SRC 경로"만 내 환경에 맞게 바꾸고 실행!
           (캐글/코랩/로컬 경로 예시는 SRC 주석 참고)
 ★ 학습 설정(epochs/imgsz/batch/seed)은 팀 전원 동일해야 비교가 공정합니다 — 건드리지 마세요.
"""
import os, glob, random, shutil
import torch
# GPU 있으면 GPU(0), 없으면 CPU 자동 선택 — LMS/코랩/캐글/로컬 어디서나 그냥 돌아가게
DEVICE = 0 if torch.cuda.is_available() else "cpu"

# ══════════════════ [설정] 여기 두 개만 바꾸세요 ══════════════════
MODEL = "yolov10s.pt"   # ★ 내가 배정받은 모델. 예: yolo11s.pt / yolov10s.pt / yolo12s.pt / rtdetr-l.pt

SRC = "/content/NordTank586x371"   # ★ 데이터 위치
#  ▶ 캐글:  SRC = "/kaggle/input/yolo-annotated-wind-turbines-586x371/NordTank586x371"
#  ▶ 코랩:  SRC = "/content/NordTank586x371"

OUT = "./split"        # split 결과가 저장될 곳 (보통 안 바꿔도 됨)
# ─────────────────── 아래 설정은 전원 동일, 건드리지 마세요 ───────────────────
EPOCHS, IMGSZ, BATCH, SEED = 50, 640, 16, 42
RATIO = (0.8, 0.1, 0.1)   # train / val / test
# ════════════════════════════════════════════════════════════════


def prepare_split():
    """데이터를 train/val/test로 나눠 split 폴더 + data.yaml 생성.
       Dirt가 희귀(1:15)해서 'Dirt 포함 여부'로 층화 분할한다. seed 고정 → 모두 동일."""
    img_dir, lbl_dir = os.path.join(SRC, "images"), os.path.join(SRC, "labels")
    # labels.txt 는 클래스이름 파일이라 제외
    label_files = sorted(f for f in glob.glob(os.path.join(lbl_dir, "*.txt"))
                         if os.path.basename(f) != "labels.txt")
    items = []  # (base, img, lbl, has_dirt)
    for lf in label_files:
        base = os.path.splitext(os.path.basename(lf))[0]
        img = os.path.join(img_dir, base + ".png")
        if not os.path.exists(img):
            continue
        has_dirt = any(line.strip().startswith("0 ") for line in open(lf))
        items.append((base, img, lf, has_dirt))

    def split_group(group):
        g = sorted(group, key=lambda x: x[0]); random.Random(SEED).shuffle(g)
        n = len(g); a = int(n*RATIO[0]); b = int(n*RATIO[1])
        return g[:a], g[a:a+b], g[a+b:]

    dirt   = [it for it in items if it[3]]
    nodirt = [it for it in items if not it[3]]
    d = split_group(dirt); nd = split_group(nodirt)
    splits = {"train": d[0]+nd[0], "val": d[1]+nd[1], "test": d[2]+nd[2]}

    if os.path.exists(OUT): shutil.rmtree(OUT)
    for sp, rows in splits.items():
        os.makedirs(os.path.join(OUT, sp, "images"), exist_ok=True)
        os.makedirs(os.path.join(OUT, sp, "labels"), exist_ok=True)
        for base, img, lbl, _ in rows:
            shutil.copy2(img, os.path.join(OUT, sp, "images", base+".png"))
            shutil.copy2(lbl, os.path.join(OUT, sp, "labels", base+".txt"))

    yaml_path = os.path.abspath(os.path.join(OUT, "data.yaml"))
    with open(yaml_path, "w") as f:
        f.write(f"path: {os.path.abspath(OUT)}\n")
        f.write("train: train/images\nval: val/images\ntest: test/images\n")
        f.write("nc: 2\nnames:\n  0: dirt\n  1: damage\n")
    for sp in ("train","val","test"):
        rows = splits[sp]; di = sum(1 for r in rows if r[3])
        print(f"  {sp:5} {len(rows):4}장 (Dirt포함 {di})")
    return yaml_path


def main():
    print("[1/2] 데이터 split 생성...")
    data_yaml = prepare_split()
    print("      data.yaml:", data_yaml)

    print(f"[2/2] 학습 시작 — 모델: {MODEL}")
    # rtdetr 만 클래스가 다름, 나머지(yolo계열)는 전부 YOLO
    if MODEL.lower().startswith("rtdetr"):
        from ultralytics import RTDETR as Net
    else:
        from ultralytics import YOLO as Net
    # ★ 결과를 구글 드라이브에 저장 → 코랩 끊겨도 체크포인트 안 날아감
    DRIVE_RUNS = "/content/drive/MyDrive/MQ02_runs"
    run_name = MODEL.split(".")[0]
    last = os.path.join(DRIVE_RUNS, run_name, "weights", "last.pt")
    if os.path.exists(last):
        # 끊겼다 다시 옴 → 저장돼 있던 last.pt에서 이어서 학습
        print(f"[resume] 드라이브에 last.pt 있음 → 이어서 학습: {last}")
        model = Net(last)
        model.train(resume=True)
    else:
        # 처음 시작 → COCO 사전학습에서 파인튜닝
        model = Net(MODEL)
        model.train(data=data_yaml, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
                    device=DEVICE, seed=SEED, project=DRIVE_RUNS, name=run_name,
                    exist_ok=True)

    # 최종: 한 번도 안 본 test 셋으로 정직하게 평가 (클래스별 AP 포함)
    print("[+] test 셋 최종 평가...")
    metrics = model.val(data=data_yaml, split="test")
    print("    test mAP50   :", round(float(metrics.box.map50), 4))
    print("    test mAP50-95:", round(float(metrics.box.map), 4))


if __name__ == "__main__":
    main()


[1/2] 데이터 split 생성...
  train 2395장 (Dirt포함 450)
  val    299장 (Dirt포함 56)
  test   301장 (Dirt포함 57)
      data.yaml: /content/split/data.yaml
[2/2] 학습 시작 — 모델: yolov10s.pt
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.83 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/split/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynami

In [6]:
from google.colab import files

# 1. 드라이브에 안전하게 자동 저장된 폴더를 zip 파일로 압축하기
!zip -rq /content/MQ02_runs_backup.zip /content/drive/MyDrive/MQ02_runs

# 2. 브라우저를 통해 내 컴퓨터로 즉시 다운로드하기
files.download('/content/MQ02_runs_backup.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>